# California Housing: Privacy-Conscious Statistical and Machine-Learning Analysis

## tl;dr

This executable notebook evaluates five area-level research questions about recorded California housing values. It verifies the data locally, avoids row-level output, engineers density proxies safely, tests for target top-coding, and compares predictive models under **spatial nested cross-validation** (primary) and shuffled nested cross-validation (secondary).

The final executed cells generate the numerical conclusions. Results describe **area-level associations and predictive information**, not causal effects or individual-house relationships.

## Context & Methods

### Privacy-conscious analysis statement

All processing occurs locally. The notebook does not transmit data, print raw records, display exact coordinate pairs, export row-level predictions, or save transformed datasets. Geographic visualizations aggregate observations and suppress cells with fewer than five records. These controls reduce unnecessary disclosure risk but do not prove anonymity or regulatory compliance.

### Key assumptions and limitations

- Each row is treated as an area-level aggregate, not an individual property or household.
- Associations are observational and must not be interpreted causally.
- Area-level findings are subject to ecological fallacy and residual confounding.
- Spatial dependence can make shuffled validation optimistic, so spatial validation is primary.
- If the target is capped, models predict the recorded capped outcome rather than uncensored market value.

## Data

### 1. Reproducibility and package checks

In [ ]:
from pathlib import Path
import importlib.util
import platform
import warnings

CORE_PACKAGES = ["pandas", "numpy", "matplotlib", "sklearn", "scipy", "statsmodels"]
missing_packages = [p for p in CORE_PACKAGES if importlib.util.find_spec(p) is None]
if missing_packages:
    raise ImportError(
        "Missing core packages: " + ", ".join(missing_packages) +
        ". Install locally before rerunning. Example (do not run automatically): "
        "pip install pandas numpy matplotlib scikit-learn scipy statsmodels"
    )

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn
import statsmodels

SEED = 20260716
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

versions = pd.Series({
    "Python": platform.python_version(), "pandas": pd.__version__,
    "NumPy": np.__version__, "Matplotlib": matplotlib.__version__,
    "scikit-learn": sklearn.__version__, "SciPy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
}, name="version")
display(versions.to_frame())

### 2. Safe local data acquisition

In [ ]:
# If `mydata` already exists, it is used. Otherwise the local project CSV is loaded.
# Change LOCAL_CSV_PATH only when the dataset is stored elsewhere.
LOCAL_CSV_PATH = Path("data/california_housing_combined.csv")
if "mydata" in globals():
    if not isinstance(mydata, pd.DataFrame):
        raise TypeError("Existing `mydata` must be a pandas DataFrame.")
    data_source = "pre-existing local DataFrame `mydata`"
else:
    if not LOCAL_CSV_PATH.exists():
        raise FileNotFoundError(f"Local dataset not found: {LOCAL_CSV_PATH.resolve()}")
    mydata = pd.read_csv(LOCAL_CSV_PATH)
    data_source = str(LOCAL_CSV_PATH.resolve())

print(f"Local source: {data_source}")
print(f"Loaded shape: {mydata.shape[0]:,} rows × {mydata.shape[1]} columns")
print("Raw rows and exact coordinate pairs are intentionally not displayed.")

### 3. Schema and data-quality validation

In [ ]:
REQUIRED = [
    "longitude", "latitude", "housing_median_age", "total_rooms",
    "total_bedrooms", "population", "households", "median_income",
    "median_house_value",
]
missing_columns = sorted(set(REQUIRED) - set(mydata.columns))
if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

analysis_data = mydata[REQUIRED].copy(deep=True)
non_numeric = [c for c in REQUIRED if not pd.api.types.is_numeric_dtype(analysis_data[c])]
if non_numeric:
    raise TypeError(f"Expected numeric columns but found non-numeric types: {non_numeric}")

quality_summary = pd.DataFrame({
    "dtype": analysis_data.dtypes.astype(str),
    "missing_n": analysis_data.isna().sum(),
    "infinite_n": np.isinf(analysis_data).sum(),
    "unique_n": analysis_data.nunique(dropna=False),
    "minimum": analysis_data.min(),
    "median": analysis_data.median(),
    "maximum": analysis_data.max(),
    "skewness": analysis_data.skew(),
})
display(quality_summary.round(3))
print(f"Duplicate rows (aggregate count only): {analysis_data.duplicated().sum():,}")
print(f"Zero/nonpositive household denominators: {(analysis_data['households'] <= 0).sum():,}")
print(f"Zero/nonpositive room denominators: {(analysis_data['total_rooms'] <= 0).sum():,}")

**Interpretation rule.** Missingness, infinities, impossible denominators, and duplicates are evaluated before modeling. Extreme but plausible communities are retained; statistical flags alone are not evidence of error. Any imputation is performed inside model pipelines to prevent leakage.

### 4. Distribution and robust outlier diagnostics

In [ ]:
def robust_outlier_summary(frame):
    rows = []
    for col in frame.columns:
        x = frame[col].dropna().to_numpy(float)
        q1, q3 = np.quantile(x, [0.25, 0.75])
        iqr = q3 - q1
        iqr_n = int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())
        med = np.median(x)
        mad = np.median(np.abs(x - med))
        rz = np.zeros_like(x) if mad == 0 else 0.6745 * (x - med) / mad
        mad_n = int((np.abs(rz) > 3.5).sum())
        rows.append((col, iqr_n, 100 * iqr_n / len(x), mad_n, 100 * mad_n / len(x)))
    return pd.DataFrame(rows, columns=["variable", "iqr_flag_n", "iqr_flag_pct", "mad_flag_n", "mad_flag_pct"])

outlier_summary = robust_outlier_summary(analysis_data)
display(outlier_summary.round(2))
print("Treatment decision: no observations are automatically removed or winsorized.")

### 5. Target top-coding investigation

In [ ]:
target = analysis_data["median_house_value"]
target_max = float(target.max())
at_max = int(target.eq(target_max).sum())
near_500001 = int(np.isclose(target, 500001, atol=1).sum())
top_coded = (at_max / len(target) >= 0.005) or (near_500001 / len(target) >= 0.005)
topcode_summary = pd.DataFrame({
    "maximum_recorded_value": [target_max],
    "count_at_maximum": [at_max],
    "percent_at_maximum": [100 * at_max / len(target)],
    "count_near_500001": [near_500001],
    "percent_near_500001": [100 * near_500001 / len(target)],
    "empirical_top_code_signal": [top_coded],
})
display(topcode_summary.round(3))

An unusually large point mass at the maximum supports—but does not mathematically prove—a top-coding interpretation. If detected, full-data results remain primary and uncapped-only results are reported as sensitivity analyses.

### 6. Privacy-conscious exploratory figures

In [ ]:
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
                     "axes.titlesize": 10, "axes.labelsize": 9})

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), constrained_layout=True)
axes[0].hist(target, bins=40, color="#4477AA", edgecolor="white")
axes[0].set(title="Recorded housing-value distribution", xlabel="Median house value ($)", ylabel="Area count")
hb = axes[1].hexbin(analysis_data["median_income"], target, gridsize=45,
                    mincnt=5, cmap="viridis")
axes[1].set(title="Income and recorded housing value", xlabel="Median income (dataset units)", ylabel="Median house value ($)")
fig.colorbar(hb, ax=axes[1], label="Areas per hexagon")
fig.savefig(FIG_DIR / "distribution_and_income.svg", bbox_inches="tight")
fig.savefig(FIG_DIR / "distribution_and_income.png", bbox_inches="tight", dpi=300)
plt.show()

### 7. Safe feature engineering

In [ ]:
model_data = analysis_data.copy()
denominator_checks = {
    "households_nonpositive": int((model_data["households"] <= 0).sum()),
    "rooms_nonpositive": int((model_data["total_rooms"] <= 0).sum()),
}
safe_households = model_data["households"].where(model_data["households"] > 0)
safe_rooms = model_data["total_rooms"].where(model_data["total_rooms"] > 0)
model_data["population_per_household"] = model_data["population"] / safe_households
model_data["rooms_per_household"] = model_data["total_rooms"] / safe_households
model_data["bedrooms_per_household"] = model_data["total_bedrooms"] / safe_households
model_data["bedroom_to_room_ratio"] = model_data["total_bedrooms"] / safe_rooms
model_data.replace([np.inf, -np.inf], np.nan, inplace=True)

engineered = ["population_per_household", "rooms_per_household",
              "bedrooms_per_household", "bedroom_to_room_ratio"]
engineered_summary = model_data[engineered].agg(["count", "mean", "median", "std", "min", "max", "skew"]).T
engineered_summary["missing_n"] = model_data[engineered].isna().sum()
display(engineered_summary.round(3))
print("Invalid denominator counts:", denominator_checks)

### 8. Spatial-block construction

In [ ]:
# Quantile bins create local geographic blocks without outputting coordinate mappings.
lon_bin = pd.qcut(model_data["longitude"], q=8, labels=False, duplicates="drop")
lat_bin = pd.qcut(model_data["latitude"], q=8, labels=False, duplicates="drop")
spatial_groups = (lon_bin.astype(str) + "_" + lat_bin.astype(str)).astype("category").cat.codes
group_counts = pd.Series(spatial_groups).value_counts()
print(f"Spatial groups created: {group_counts.size}")
print(f"Group-size range: {group_counts.min():,}–{group_counts.max():,}; median {group_counts.median():,.0f}")
print("Exact coordinate-to-group assignments are intentionally not displayed or exported.")

## Results

### Research Question 1 — Income and recorded housing value

In [ ]:
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

x_income = model_data["median_income"]
y_value = model_data["median_house_value"]
pearson = stats.pearsonr(x_income, y_value)
spearman = stats.spearmanr(x_income, y_value)
linear_income = sm.OLS(y_value, sm.add_constant(x_income)).fit(cov_type="HC3")
bp = het_breuschpagan(linear_income.resid, linear_income.model.exog)

income_results = pd.DataFrame({
    "estimate": [pearson.statistic, spearman.statistic, linear_income.params["median_income"]],
    "ci_low": [pearson.confidence_interval().low, np.nan, linear_income.conf_int().loc["median_income", 0]],
    "ci_high": [pearson.confidence_interval().high, np.nan, linear_income.conf_int().loc["median_income", 1]],
    "p_value": [pearson.pvalue, spearman.pvalue, linear_income.pvalues["median_income"]],
}, index=["Pearson r", "Spearman rho", "OLS slope ($ per income unit), HC3"])
display(income_results.round(4))
print(f"Simple linear-model R²: {linear_income.rsquared:.3f}")
print(f"Breusch–Pagan p-value: {bp[1]:.3g}; HC3 robust standard errors are reported.")

uncapped = ~y_value.eq(target_max) if top_coded else pd.Series(True, index=y_value.index)
uncapped_model = sm.OLS(y_value[uncapped], sm.add_constant(x_income[uncapped])).fit(cov_type="HC3")
print(f"Income slope sensitivity (excluding maximum-coded target): ${uncapped_model.params['median_income']:,.0f} per income unit")

**Statistical logic.** The null hypotheses are zero Pearson correlation, zero rank correlation, and a zero income slope. Pearson/OLS summarize linear association; Spearman is robust to monotonic nonlinearity. HC3 uncertainty is used because heteroskedasticity is plausible. With 20,000 areas, effect magnitude and predictive fit matter more than p-values alone.

### Shared leakage-safe modeling utilities

In [ ]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

TARGET = "median_house_value"
BASE_FEATURES = [c for c in model_data.columns if c != TARGET]
X = model_data[BASE_FEATURES]
y = model_data[TARGET]

def numeric_pipeline(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

MODEL_SPECS = {
    "Median baseline": (numeric_pipeline(DummyRegressor(strategy="median")), {}),
    "Ridge": (numeric_pipeline(Ridge(), scale=True), {"model__alpha": [1.0, 10.0]}),
    "Extra Trees": (numeric_pipeline(ExtraTreesRegressor(n_estimators=80, min_samples_leaf=2,
                    random_state=SEED, n_jobs=-1)), {"model__max_features": [0.7, 1.0]}),
    "Histogram Gradient Boosting": (numeric_pipeline(HistGradientBoostingRegressor(
                    random_state=SEED, max_iter=150)), {"model__learning_rate": [0.05, 0.1]}),
}

def score_predictions(y_true, pred):
    return {"MAE": mean_absolute_error(y_true, pred),
            "RMSE": mean_squared_error(y_true, pred) ** 0.5,
            "R2": r2_score(y_true, pred),
            "MedianAE": median_absolute_error(y_true, pred)}

def nested_evaluate(X_data, y_data, outer_cv, protocol, groups=None):
    records, fitted = [], {}
    splits = list(outer_cv.split(X_data, y_data, groups))
    for model_name, (estimator, grid) in MODEL_SPECS.items():
        for fold, (train_idx, test_idx) in enumerate(splits, 1):
            X_train, X_test = X_data.iloc[train_idx], X_data.iloc[test_idx]
            y_train, y_test = y_data.iloc[train_idx], y_data.iloc[test_idx]
            inner = KFold(n_splits=3, shuffle=True, random_state=SEED + fold)
            if grid:
                fitted_model = GridSearchCV(clone(estimator), grid, scoring="neg_mean_absolute_error",
                                            cv=inner, n_jobs=-1).fit(X_train, y_train).best_estimator_
            else:
                fitted_model = clone(estimator).fit(X_train, y_train)
            metrics = score_predictions(y_test, fitted_model.predict(X_test))
            records.append({"protocol": protocol, "model": model_name, "fold": fold, **metrics})
            fitted[(protocol, model_name, fold)] = (fitted_model, train_idx, test_idx)
    return pd.DataFrame(records), fitted

### Research Question 2 — Incremental geographic information

In [ ]:
feature_sets = {
    "A: income only": ["median_income"],
    "B: income + non-geographic housing": [c for c in BASE_FEATURES if c not in ["longitude", "latitude"]],
    "C: B + coordinates": BASE_FEATURES,
}
spatial_cv = GroupKFold(n_splits=5)
location_rows = []
for label, cols in feature_sets.items():
    fold_mae = []
    for train_idx, test_idx in spatial_cv.split(model_data[cols], y, spatial_groups):
        model = numeric_pipeline(Ridge(alpha=10), scale=True).fit(model_data.iloc[train_idx][cols], y.iloc[train_idx])
        fold_mae.append(mean_absolute_error(y.iloc[test_idx], model.predict(model_data.iloc[test_idx][cols])))
    location_rows.append((label, np.mean(fold_mae), np.std(fold_mae, ddof=1)))
location_comparison = pd.DataFrame(location_rows, columns=["model", "spatial_MAE_mean", "spatial_MAE_sd"])
display(location_comparison.round(0))

Model C's change relative to Model B estimates the additional *linear predictive information* supplied by coordinates after adjustment. It is not a causal effect of location. Nonlinear geographic information is evaluated by the tree-based models below.

### Research Question 3 — Density and crowding proxies

In [ ]:
rq3_features = ["population_per_household", "rooms_per_household", "bedrooms_per_household",
                "bedroom_to_room_ratio", "median_income", "longitude", "latitude", "housing_median_age"]
rq3_frame = model_data[rq3_features + [TARGET]].replace([np.inf, -np.inf], np.nan).dropna()
rq3_X = sm.add_constant(rq3_frame[rq3_features])
rq3_fit = sm.OLS(rq3_frame[TARGET], rq3_X).fit(cov_type="HC3")
proxy_rows = []
for name in engineered:
    proxy_rows.append({"proxy": name, "estimate": rq3_fit.params[name],
                       "ci_low": rq3_fit.conf_int().loc[name, 0], "ci_high": rq3_fit.conf_int().loc[name, 1],
                       "p_value_raw": rq3_fit.pvalues[name]})
proxy_results = pd.DataFrame(proxy_rows)
proxy_results["p_value_bonferroni"] = np.minimum(proxy_results["p_value_raw"] * len(engineered), 1)
display(proxy_results.round(4))
print(f"Adjusted model R²: {rq3_fit.rsquared:.3f}; HC3 robust uncertainty and Bonferroni-adjusted p-values shown.")

These coefficients are adjusted area-level associations. The engineered measures are imperfect proxies—not direct measurements of physical crowding—and coefficients may be unstable when ratios are correlated.

### Research Question 4 — Housing-stock age and nonlinearity

In [ ]:
age_controls = ["median_income", "longitude", "latitude", "population_per_household", "rooms_per_household"]
age_frame = model_data[["housing_median_age", TARGET] + age_controls].dropna()
linear_formula_X = sm.add_constant(age_frame[["housing_median_age"] + age_controls])
quadratic_X = linear_formula_X.copy()
quadratic_X["housing_median_age_squared"] = age_frame["housing_median_age"] ** 2
age_linear = sm.OLS(age_frame[TARGET], linear_formula_X).fit(cov_type="HC3")
age_quadratic = sm.OLS(age_frame[TARGET], quadratic_X).fit(cov_type="HC3")
age_results = pd.DataFrame({
    "model": ["linear age", "quadratic age"],
    "adjusted_R2": [age_linear.rsquared_adj, age_quadratic.rsquared_adj],
    "AIC": [age_linear.aic, age_quadratic.aic],
})
display(age_results.round(3))
print("Linear age coefficient and HC3 95% CI:",
      round(age_linear.params["housing_median_age"], 1),
      tuple(np.round(age_linear.conf_int().loc["housing_median_age"], 1)))
print("Quadratic-term p-value:", f"{age_quadratic.pvalues['housing_median_age_squared']:.3g}")

age_grid = np.linspace(age_frame["housing_median_age"].quantile(.01), age_frame["housing_median_age"].quantile(.99), 100)
reference = age_frame[age_controls].median()
plot_X = pd.DataFrame({c: reference[c] for c in age_controls}, index=range(len(age_grid)))
plot_X.insert(0, "housing_median_age", age_grid)
plot_X = sm.add_constant(plot_X, has_constant="add")
plot_X["housing_median_age_squared"] = age_grid ** 2
pred_frame = age_quadratic.get_prediction(plot_X[quadratic_X.columns]).summary_frame()
fig, ax = plt.subplots(figsize=(3.5, 2.8), constrained_layout=True)
ax.plot(age_grid, pred_frame["mean"], color="#228833")
ax.fill_between(age_grid, pred_frame["mean_ci_lower"], pred_frame["mean_ci_upper"], color="#228833", alpha=.2)
ax.set(title="Adjusted age–value relationship", xlabel="Housing median age (years)", ylabel="Predicted recorded value ($)")
fig.savefig(FIG_DIR / "adjusted_age_effect.svg", bbox_inches="tight")
fig.savefig(FIG_DIR / "adjusted_age_effect.png", bbox_inches="tight", dpi=300)
plt.show()

### Research Question 5 — Nested model comparison

In [ ]:
spatial_results, spatial_fits = nested_evaluate(X, y, GroupKFold(n_splits=5), "Spatial", spatial_groups)
random_results, random_fits = nested_evaluate(X, y, KFold(n_splits=5, shuffle=True, random_state=SEED), "Random")
cv_results = pd.concat([spatial_results, random_results], ignore_index=True)
model_summary = cv_results.groupby(["protocol", "model"])[["MAE", "RMSE", "R2", "MedianAE"]].agg(["mean", "std"])
display(model_summary.round(1))

The same five outer folds are reused across models within each protocol. Hyperparameters are selected only inside the corresponding training partition. Spatial results are primary; random results quantify how much shuffled evaluation may overstate geographic generalization.

### Model interpretation

In [ ]:
spatial_means = spatial_results.groupby("model")["MAE"].mean().sort_values()
best_name = spatial_means.index[0]
best_model, _, best_test_idx = spatial_fits[("Spatial", best_name, 1)]
perm = permutation_importance(best_model, X.iloc[best_test_idx], y.iloc[best_test_idx],
                              scoring="neg_mean_absolute_error", n_repeats=5, random_state=SEED, n_jobs=-1)
importance = pd.DataFrame({"feature": BASE_FEATURES, "MAE_importance": perm.importances_mean,
                           "importance_sd": perm.importances_std}).sort_values("MAE_importance", ascending=False)
display(importance.head(10).round(1))

fig, ax = plt.subplots(figsize=(4.2, 3.2), constrained_layout=True)
top = importance.head(10).sort_values("MAE_importance")
ax.barh(top["feature"], top["MAE_importance"], xerr=top["importance_sd"], color="#CC6677")
ax.set(title=f"Held-out permutation importance: {best_name}", xlabel="Increase in MAE after permutation ($)")
fig.savefig(FIG_DIR / "permutation_importance.svg", bbox_inches="tight")
fig.savefig(FIG_DIR / "permutation_importance.png", bbox_inches="tight", dpi=300)
plt.show()

Permutation importance measures held-out predictive contribution, not causality. Correlated features can share or obscure importance, and estimates may vary across spatial folds.

### Sensitivity analysis for target top-coding

In [ ]:
best_estimator_template = MODEL_SPECS[best_name][0]
uncapped_mask = ~y.eq(target_max) if top_coded else pd.Series(True, index=y.index)
sensitivity_scores = []
for label, mask in [("Full recorded target", pd.Series(True, index=y.index)),
                    ("Maximum-coded observations excluded", uncapped_mask)]:
    local_X, local_y = X.loc[mask].reset_index(drop=True), y.loc[mask].reset_index(drop=True)
    local_groups = pd.Series(spatial_groups)[mask.to_numpy()].reset_index(drop=True)
    fold_scores = []
    for tr, te in GroupKFold(n_splits=5).split(local_X, local_y, local_groups):
        fitted = clone(best_estimator_template).fit(local_X.iloc[tr], local_y.iloc[tr])
        fold_scores.append(mean_absolute_error(local_y.iloc[te], fitted.predict(local_X.iloc[te])))
    sensitivity_scores.append((label, np.mean(fold_scores), np.std(fold_scores, ddof=1), len(local_y)))
sensitivity_table = pd.DataFrame(sensitivity_scores, columns=["analysis", "spatial_MAE_mean", "spatial_MAE_sd", "n"])
display(sensitivity_table.round(1))

## Takeaways

In [ ]:
best_spatial = spatial_results.groupby("model").agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"), R2=("R2", "mean")).sort_values("MAE")
best_random = random_results.groupby("model").agg(MAE=("MAE", "mean"), R2=("R2", "mean")).sort_values("MAE")
geo_gain = location_comparison.set_index("model").loc["B: income + non-geographic housing", "spatial_MAE_mean"] - location_comparison.set_index("model").loc["C: B + coordinates", "spatial_MAE_mean"]
age_nonlinear_better = age_quadratic.aic < age_linear.aic

print("1. INCOME:")
print(f"   Median income has a Pearson correlation of {pearson.statistic:.3f} and Spearman correlation of {spearman.statistic:.3f} with recorded value.")
print(f"   The simple model explains {linear_income.rsquared:.1%} of area-level variation; this is association, not causation.")
print("2. LOCATION:")
print(f"   Adding coordinates to the adjusted linear feature set changes spatial MAE by ${geo_gain:,.0f}; nonlinear models test richer geographic structure.")
print("3. DENSITY/CROWDING PROXIES:")
strongest_proxy = proxy_results.iloc[proxy_results["estimate"].abs().argmax()]
print(f"   The largest raw adjusted coefficient among the four proxies is for {strongest_proxy['proxy']}; interpretation remains scale-dependent and noncausal.")
print("4. HOUSING AGE:")
print(f"   The quadratic specification {'improves' if age_nonlinear_better else 'does not improve'} AIC relative to the linear specification, supporting a {'nonlinear' if age_nonlinear_better else 'primarily linear'} adjusted pattern.")
print("5. PREDICTION:")
print(f"   Best spatial model: {best_spatial.index[0]} (MAE ${best_spatial.iloc[0]['MAE']:,.0f}, RMSE ${best_spatial.iloc[0]['RMSE']:,.0f}, R² {best_spatial.iloc[0]['R2']:.3f}).")
print(f"   Best shuffled model: {best_random.index[0]} (MAE ${best_random.iloc[0]['MAE']:,.0f}, R² {best_random.iloc[0]['R2']:.3f}).")
print("\nSUPPORTED: aggregate descriptions, adjusted associations, and out-of-sample predictive comparisons.")
print("UNSUPPORTED: causal effects, individual-house conclusions, uncensored values above any detected cap, or universal geographic generalization.")

### Final limitations and preprocessing record

- The original source was preserved; no automatic outlier deletion or winsorization was applied.
- Invalid engineered divisions were converted to missing values and handled by training-fold median imputation.
- Spatial blocks are an approximation to geographic separation, not a complete spatial statistical model.
- Coordinate-based associations may stand in for unmeasured coastal access, urbanization, regulation, amenities, employment access, or other omitted factors.
- Area-level medians and totals cannot support individual-level conclusions.
- If top-coding is present, ordinary regression cannot infer the uncensored upper tail.
- Publication-ready figures use accessible colors, vector export, and 300-DPI PNG fallbacks; journal-specific formatting may still be required.

Generated aggregate files are listed below. No row-level analytical outputs are saved.

In [ ]:
generated_files = sorted(str(p) for p in FIG_DIR.glob("*") if p.suffix.lower() in {".png", ".svg"})
print("Generated aggregate figures:")
for file_name in generated_files:
    print(" -", file_name)
print(f"\nReproducibility seed: {SEED}")
print(f"Data source used locally: {data_source}")
print("Notebook execution should be repeated after any data or package-version change.")